# [실습] LangChain 기본 구조


LangChain을 활용하여 파이썬 프로그램 내에서 LLM을 활용해 보겠습니다.   

---



## 라이브러리 설치  
`langchain_openai`, `langchain_google_genai` 등의 라이브러리를 이용해 provider별 모델을 활용합니다.     
`langchain_ollama`, `langchain_huggingface` 를 통해 오픈 모델을 연동할 수도 있습니다.

In [ ]:
%pip install langchain openai langchain_openai rich dotenv -q

## API 키 등록 + dotenv로 환경 변수 불러오기


1. `.env` 파일 만들기    
좌측의 파일 탭에서 `.env` 파일을 만들어 주세요.   
(숨김 파일 표시 체크가 필요합니다.)


2. OpenAI API 키    
학습 시트에 있는 키를 `OPENAI_API_KEY="sk-..."` 형식으로 저장하세요.

생성한 `.env` 파일은 이후 실습에서 계속 사용하므로, 다운로드해서 보관하는 것이 좋습니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
import openai
client = openai.OpenAI()

# API 키 검증하기
try:
    client.models.list()
    print("OPENAI_API_KEY가 정상적으로 설정되어 있습니다.")
except openai.AuthenticationError:
    raise Exception("API 키가 유효하지 않습니다!")

## LLM

LLM은 `ChatOpenAI`, `ChatGoogleGenerativeAI`와 같은 클래스로 불러올 수 있습니다.

In [ ]:
from langchain_openai import ChatOpenAI

gpt5  = ChatOpenAI(model='gpt-5.2', reasoning_effort='low', verbosity='medium', max_tokens=32768)

In [ ]:
from rich import print as rprint
# 복잡한 구조는 rich를 통해 출력

rprint(gpt5)

## Prompt

LLM에 입력할 프롬프트는 다음의 방법으로 전달됩니다.

1. 단순 문자열
2. 랭체인 메시지 클래스
3. 프롬프트 템플릿과 입력 변수


### 1. 단순 문자열   

LLM은 `invoke()`를 통해 실행합니다.

In [ ]:
question = '''
프롬프트 엔지니어링에서 가장 중요한 5개 원칙을 예시를 포함하여 각각 150자 이내로 설명하세요.
'''

response = gpt5.invoke(question)
response

출력 형식은 AIMessage 클래스입니다.   
입력 문자열은 HumanMessage 클래스로 변환되어 전달됩니다.

In [ ]:
rprint(response)

메타데이터를 통해 토큰 사용량도 확인할 수 있습니다.

In [ ]:
response.usage_metadata

답변 본문만 필요할 때는 `.text`로 접근합니다.

In [ ]:
print(response.text)

batch를 통해 여러 개의 입력을 병렬적으로 처리할 수도 있습니다.

In [ ]:
topics = ['LLM이 무엇의 약자인가요? 20단어 이내로 답변하세요.',
          'LLM이랑 GPT랑 다른 건가요? 20단어 이내로 답변하세요.',
          'BERT와 GPT는 뭐가 다른가요? 20단어 이내로 답변하세요.']
results = gpt5.batch(topics)
results

Human Message 이외에도, LLM은 챗봇의 작동 방식을 결정하는 System Message를 지원합니다.

System Message는 보통 전체 대화의 첫 번째로 들어갑니다.

### 2. Message 클래스 전달하기   
클래스를 직접 생성하고 전달합니다.

In [ ]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage

question = ''

messages = [
    SystemMessage('당신은 매우 논리적이고, 다양한 관점을 고려합니다.'),
    HumanMessage(question)
]

if question:
    response = gpt5.invoke(messages)
    rprint(response)

AIMessage를 함께 전달하는 방식으로, 멀티-턴 대화를 수행할 수 있습니다.

In [ ]:
followup_msg = HumanMessage('')

if followup_msg.content:

    new_messages = messages+[response, followup_msg]

    response2 = gpt5.invoke(new_messages)
    rprint(response2)

### 3. Prompt Template

프롬프트 템플릿을 사용하면, 정해진 템플릿에 입력 변수의 공간을 설정하여, 프롬프트의 포맷을 재사용할 수 있습니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

System, AI 등의 메시지를 포함하기 위해서는 ChatPromptTemplate를 사용합니다.


프롬프트 템플릿과 LLM은 체인(Chain)을 통해 연결합니다.

In [ ]:
chat_prompt = ChatPromptTemplate([
    ("system", '당신은 항상 이모지로만 대답합니다.'),
    ("user", '{topic}에 대해 설명해주세요.')
    # 역할은 4개 (user = human), (ai = assistant)
]
)

chain = chat_prompt | gpt5
# 왼쪽에서 오른쪽으로 순차적 실행되는 Sequence 구조

In [ ]:
chain.invoke("RAG")

### 멀티모달 프롬프트 전달하기

멀티모달 모델은 이미지의 URL이나 실제 파일을 프롬프트에 전달할 수 있습니다.

In [ ]:
import base64
import httpx

image_url = "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/scones.jpg"
save_path = "scones.jpg"

with httpx.Client(timeout=30.0) as http_client:
    with http_client.stream("GET", image_url) as r:
        r.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in r.iter_bytes():
                f.write(chunk)

# 파일 크기 체크
print("saved:", save_path, "bytes:", os.path.getsize(save_path))


이미지 URL을 전달합니다.

In [ ]:
message = HumanMessage(
    content=[
        {"type": "text", "text": "이 그림에 대해 설명해주세요."},
        {"type": "image", "url": image_url},
    ]
)

ai_msg = gpt5.invoke([message])
ai_msg.text

오프라인 파일은 base64 인코딩을 거쳐야 합니다.

In [ ]:
with open('./scones.jpg', 'rb') as image_file:
    image_data = base64.b64encode(image_file.read()).decode('utf-8')

In [ ]:
message = HumanMessage([
        {"type": "text", "text": "이 사진에 보이는 사물의 종류와 개수를 모두 찾아서 표 형태로 출력하세요."},
        {"type": "image", "base64": image_data, "mime_type": "image/jpeg"},
    ]
)

ai_msg = gpt5.invoke([message])
ai_msg.text